In [2]:
import cv2
import numpy as np

In [6]:
cap = cv2.VideoCapture(0) #첫번째 카메라 사용
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")

while True:
    (ret,frame) = cap.read()
    if not ret:
        print("프레임 오류")
        break

    flip_frame = cv2.flip(frame,1) #좌우 반전
    (height, width, _) = flip_frame.shape
    (center_x, center_y) = (width//2, height//2)
    roi = flip_frame[center_y - 150:center_y + 150, center_x - 150:center_x + 150]
    cv2.rectangle(flip_frame,(center_x-150,center_y-150),(center_x + 150, center_y+ 150),(0,0,255),2)

    cv2.imshow("Webcam", flip_frame)

    key = cv2.waitKey(1) & 0xFF

    if key in (ord('c'), ord('C')):
        gray_image = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray_image = np.flip(gray_image, axis=1)
        cv2.imwrite('gray_image.png', gray_image)
        gaussian_blur = cv2.GaussianBlur(gray_image, (5, 5), 3)
        (_, otsh_thresh) = cv2.threshold(gaussian_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cv2.imshow("OTSU",otsh_thresh)
        kernel = np.ones((5,5), np.uint8)
        erosion = cv2.erode(otsh_thresh, kernel, iterations=5)
        cv2.imshow("EROSION", erosion)
        cv2.imwrite("digit_binary_image.png",erosion)

        img = erosion
        (h, w) = img.shape[:2]
        digit_mask = np.uint8(img < 128) * 255
        contours, _ = cv2.findContours(digit_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            contour = max(contours, key=cv2.contourArea)
            x, y, crop_w, crop_h = cv2.boundingRect(contour)
            padding = 10
            x1 = max(0, x - padding)
            y1 = max(0, y - padding)
            x2 = min(w, x + crop_w + padding)
            y2 = min(h, y + crop_h + padding)
            crop = img[y1:y2, x1:x2]
        else:
            crop = img
        cv2.imshow("CROP", crop)
        reversed_image = cv2.bitwise_not(crop)
        cv2.imshow("REVERSED_IMAGE", reversed_image)
        cv2.imwrite("IMAGE_FOR_TEST.png", reversed_image)


    if cv2.waitKey(30) == 27:
        break
cap.release()
cv2.destroyAllWindows()
